# Differential Equations — Session 37
## Section 8.2: Homogeneous Linear Systems

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. derive the eigenvalue equation from the trial solution $e^{\lambda t}\mathbf K$;
2. solve systems with distinct real eigenvalues;
3. handle repeated eigenvalues with enough eigenvectors;
4. construct generalized eigenvector solutions for defective matrices;
5. form real solutions from complex eigenpairs;
6. interpret trajectories and phase portraits;
7. connect eigenvalue real parts with growth, decay, and oscillation.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–16 min | Eigenvalue derivation |
| 16–38 min | Distinct real eigenvalues |
| 38–55 min | Repeated eigenvalues |
| 55–72 min | Defective matrices |
| 72–86 min | Complex eigenvalues and phase portraits |
| 86–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad_vec
from scipy.linalg import expm, eig
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def solve_linear_system(A, x0, t_span=(-5, 5), points=1200, forcing=None):
    A = np.asarray(A, dtype=float)
    x0 = np.asarray(x0, dtype=float)
    t_eval = np.linspace(t_span[0], t_span[1], points)

    if forcing is None:
        def rhs(t, x):
            return A @ x
    else:
        def rhs(t, x):
            return A @ x + np.asarray(forcing(t), dtype=float)

    return solve_ivp(rhs, t_span, x0, t_eval=t_eval, rtol=1e-9, atol=1e-11)

def vector_field(A, xlim=(-4, 4), ylim=(-4, 4), density=21):
    A = np.asarray(A, dtype=float)
    x = np.linspace(*xlim, density)
    y = np.linspace(*ylim, density)
    X, Y = np.meshgrid(x, y)
    U = A[0,0]*X + A[0,1]*Y
    V = A[1,0]*X + A[1,1]*Y
    speed = np.sqrt(U**2 + V**2)
    U = np.divide(U, speed, out=np.zeros_like(U), where=speed>1e-12)
    V = np.divide(V, speed, out=np.zeros_like(V), where=speed>1e-12)
    plt.quiver(X, Y, U, V, speed)
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel("x")
    plt.ylabel("y")

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Principle 8.2-A — Exponential vector trial

Assume

$$
\mathbf X=e^{\lambda t}\mathbf K.
$$

Then

$$
(A-\lambda I)\mathbf K=\mathbf 0.
$$

A nonzero vector exists only when

$$
\det(A-\lambda I)=0.
$$

### Definition 8.2-B — Eigenvalue and eigenvector

A scalar $\lambda$ satisfying the characteristic equation is an eigenvalue. A nonzero vector $\mathbf K$ satisfying

$$
A\mathbf K=\lambda\mathbf K
$$

is an associated eigenvector.

### Theorem 8.2-C — Distinct real eigenvalues

If $A$ has $n$ distinct real eigenvalues with eigenvectors $\mathbf K_1,\ldots,\mathbf K_n$, then

$$
\mathbf X
=
\sum_{j=1}^n
c_j e^{\lambda_jt}\mathbf K_j.
$$

### Proposition 8.2-D — Repeated eigenvalue with enough eigenvectors

If an eigenvalue of algebraic multiplicity $m$ has $m$ independent eigenvectors, each produces an ordinary exponential solution.

### Theorem 8.2-E — Defective repeated eigenvalue

If $\lambda$ has one eigenvector $\mathbf K$ and a generalized eigenvector $\mathbf P$ satisfying

$$
(A-\lambda I)\mathbf P=\mathbf K,
$$

then two independent solutions are

$$
e^{\lambda t}\mathbf K,
$$

$$
e^{\lambda t}(t\mathbf K+\mathbf P).
$$

### Theorem 8.2-F — Complex eigenvalue

For real $A$, if $\lambda=\alpha+i\beta$ with eigenvector $\mathbf K=\mathbf B_1+i\mathbf B_2$, then real independent solutions are obtained from the real and imaginary parts of

$$
e^{(\alpha+i\beta)t}\mathbf K.
$$

### Classroom Checkpoint — Read the Eigenvalues

A real $2\times2$ system has eigenvalues $-1\pm3i$. What qualitative behavior occurs near the origin?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Distinct real eigenvalues

Take

$$
A=
\begin{pmatrix}
2&1\\
1&2
\end{pmatrix}.
$$

The eigenvalues are $3$ and $1$, with eigenvectors along the lines $y=x$ and $y=-x$.

In [ ]:
A = np.array([[2,1],[1,2]], dtype=float)
vals, vecs = np.linalg.eig(A)
print("eigenvalues:", vals)
print("eigenvectors:")
print(vecs)

vector_field(A, (-4,4), (-4,4))
for x0 in ([1,0], [0,1], [1,-1], [-1,-2], [2,1]):
    sol = solve_linear_system(A, x0, (-2, 1.2), 800)
    plt.plot(sol.y[0], sol.y[1])
plt.title("Distinct positive eigenvalues")
plt.show()

### Interactive real-eigenvalue phase portrait

In [ ]:
def real_eigenvalue_explorer(lambda1=-1.0, lambda2=2.0, angle=25.0):
    theta = np.deg2rad(angle)
    P = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    A = P @ np.diag([lambda1, lambda2]) @ np.linalg.inv(P)

    vector_field(A, (-4,4), (-4,4))
    initials = ([3,0],[0,3],[-3,1],[1,-3],[2,2],[-2,-2])
    for x0 in initials:
        sol = solve_linear_system(A, x0, (-4,4), 1000)
        plt.plot(sol.y[0], sol.y[1])
    plt.title(fr"$\lambda_1={lambda1:.2f},\lambda_2={lambda2:.2f}$")
    plt.show()
    print("A =")
    print(A)

if WIDGETS_AVAILABLE:
    interact(
        real_eigenvalue_explorer,
        lambda1=FloatSlider(min=-3, max=3, step=0.25, value=-1),
        lambda2=FloatSlider(min=-3, max=3, step=0.25, value=2),
        angle=FloatSlider(min=0, max=90, step=5, value=25)
    )
else:
    real_eigenvalue_explorer()

## 2. Repeated eigenvalue with two eigenvectors

If

$$
A=
\begin{pmatrix}
-2&0\\
0&-2
\end{pmatrix},
$$

every nonzero vector is an eigenvector. All trajectories move radially toward the origin.

In [ ]:
A = -2*np.eye(2)
vector_field(A)
for x0 in ([3,0], [2,2], [-3,1], [0,-3]):
    sol = solve_linear_system(A, x0, (0, 3), 400)
    plt.plot(sol.y[0], sol.y[1])
plt.title("Repeated eigenvalue with two eigenvectors")
plt.show()

## 3. Defective repeated eigenvalue

For

$$
A=
\begin{pmatrix}
-1&1\\
0&-1
\end{pmatrix},
$$

there is one eigenvector. The factor $t e^{-t}$ bends trajectories before decay.

In [ ]:
A = np.array([[-1,1],[0,-1]], dtype=float)
vector_field(A)
for x0 in ([3,0], [2,2], [-3,1], [0,-3], [1,-2]):
    sol = solve_linear_system(A, x0, (0, 8), 800)
    plt.plot(sol.y[0], sol.y[1])
plt.title("Defective stable node")
plt.show()

In [ ]:
def jordan_explorer(lambda_value=-1.0, shear=1.0):
    A = np.array([[lambda_value, shear], [0, lambda_value]], dtype=float)
    t = np.linspace(0, 8, 700)
    X = np.array([expm(A*ti) @ np.array([0,1]) for ti in t])

    plt.plot(t, X[:,0], label="x(t)")
    plt.plot(t, X[:,1], label="y(t)")
    plt.legend()
    plt.title("Polynomial-exponential behavior from a Jordan block")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        jordan_explorer,
        lambda_value=FloatSlider(min=-2, max=1, step=0.1, value=-1),
        shear=FloatSlider(min=-3, max=3, step=0.25, value=1)
    )
else:
    jordan_explorer()

## 4. Complex eigenvalues

For

$$
A=
\begin{pmatrix}
\alpha&-\beta\\
\beta&\alpha
\end{pmatrix},
$$

the eigenvalues are $\alpha\pm i\beta$.

- $\alpha<0$: inward spiral;
- $\alpha=0$: center;
- $\alpha>0$: outward spiral.

In [ ]:
def complex_pair_explorer(alpha=-0.3, beta=2.0):
    A = np.array([[alpha, -beta], [beta, alpha]], dtype=float)
    vector_field(A, (-4,4), (-4,4))
    for x0 in ([3,0],[2,2],[-3,1],[0,-3]):
        sol = solve_linear_system(A, x0, (0, 15), 1200)
        plt.plot(sol.y[0], sol.y[1])
    plt.title(fr"Eigenvalues $\alpha\pm i\beta={alpha:.2f}\pm {beta:.2f}i$")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        complex_pair_explorer,
        alpha=FloatSlider(min=-1, max=1, step=0.1, value=-0.3),
        beta=FloatSlider(min=0.25, max=4, step=0.25, value=2)
    )
else:
    complex_pair_explorer()

## 5. Orientation of rotation

At the point $(1,0)$,

$$
\mathbf X'=A\mathbf X
$$

shows whether the flow initially points upward or downward. This determines clockwise or counterclockwise rotation.

In [ ]:
for beta in [2, -2]:
    A = np.array([[0, -beta], [beta, 0]], dtype=float)
    velocity = A @ np.array([1.0, 0.0])
    print("beta =", beta, "velocity at (1,0) =", velocity)

## Optional extension — A three-dimensional system

Distinct eigenvalues and eigenvectors work exactly the same way in three dimensions.

In [ ]:
A3 = np.array([[-1,0,0],[0,-2,1],[0,0,-3]], dtype=float)
vals, vecs = np.linalg.eig(A3)
print("eigenvalues:", vals)
print("eigenvectors:")
print(vecs)

## Classroom Checkpoint — Exit Check

For

$$
A=
\begin{pmatrix}
0&-4\\
1&0
\end{pmatrix},
$$

find the eigenvalues.

> Pause here. Let students commit to an answer before running the next cell.